# CMRI_Model — PI composite figures

Preserves the original scientific calculations, warm palette and composite layouts. Editable SVG text uses Arial (a compatible installed font is used for rendering if necessary). Main figure titles are 15 pt; all other text is 12 pt. PCA uses a shared continuous scale for each descriptor, with low values light and high values dark, matching the reference figure's color direction. The stored 3D conformer displays physical X/Y/Z axes, ticks and grid in Å. No new conformer generation or inference. Stable output directory and filenames.

In [ ]:
from pathlib import Path
import json, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from IPython.display import display
from PIL import Image
PROJECT=Path('/home/ypan/heart_liver/model_train/heart-molecular-pretraining')
GENERAL=Path('/project/yli4/ypan/heart_evaluation/general_audit_10k_v2')
VALIDATION=Path('/home/ypan/heart_liver/model_train/validation')
RUN=GENERAL/'checkpoint_evaluation/full_10k_batch8'
TRAIN=GENERAL/'checkpoint_evaluation/train_10k_batch8'
BENCH=GENERAL/'chemistry_slide_evidence/independent_property_benchmarks/benchmark_all_runs.csv'
UGI=VALIDATION/'evidence/ugi/ugi_priority1_interactive_evidence'
LNP=VALIDATION/'evidence/lnpdb/ugi_lnpdb_priority2_evidence'
OUT=PROJECT/'summary_figures/cmri_model_pi_composites'
OUT.mkdir(parents=True,exist_ok=True)
from matplotlib import font_manager
# Arial is the target font for editable SVGs in Illustrator.
# Liberation Sans / Arimo are metrically compatible alternatives for Linux rendering.
SVG_FONT = 'Arial'
FONT_SIZE = 12
MAIN_TITLE_SIZE = 15
installed_fonts = {f.name for f in font_manager.fontManager.ttflist}
RENDER_FONT = next((f for f in ['Arial', 'Liberation Sans', 'Arimo', 'DejaVu Sans'] if f in installed_fonts), 'DejaVu Sans')
plt.rcParams.update({'svg.fonttype':'none','font.family':RENDER_FONT,'font.size':FONT_SIZE,'axes.spines.top':False,'axes.spines.right':False})
print(f'Rendering font: {RENDER_FONT}; editable SVG font: {SVG_FONT}')
if RENDER_FONT == 'DejaVu Sans':
    print('NOTE: Arial-compatible font not found; check spacing in Illustrator after substitution.')

def normalize_svg_font(path):
    # Matplotlib exports either font-family or the CSS font shorthand.
    # Replace the specific rendering family, leaving all text and paths editable.
    path = Path(path)
    svg = path.read_text(encoding='utf-8')
    if RENDER_FONT != SVG_FONT:
        svg = svg.replace(RENDER_FONT, SVG_FONT)
    # Preserve clip paths; never down-convert to SVG Tiny.
    path.write_text(svg, encoding='utf-8')
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.ticker import FuncFormatter
# Warm palette, inspired by the supplied reference; dark muted plum only as accent.
PALETTE=['#F4E4A9','#EDC65F','#E6A34C','#DC8051','#CF5B50','#B54A55','#7C546D']
RED='#B54A55'; GOLD='#E6A34C'; PLUM='#7C546D'
WARM=LinearSegmentedColormap.from_list('cmri_warm_high_dark',['#F4E4A9','#EDC65F','#E6A34C','#DC8051','#CF5B50','#B54A55','#7C546D']) # low = light, high = dark
plt.rcParams.update({k: FONT_SIZE for k in ['font.size','axes.labelsize','axes.titlesize','xtick.labelsize','ytick.labelsize','legend.fontsize','legend.title_fontsize','figure.titlesize']})
plt.rcParams['axes.titleweight']='bold'
plt.rcParams['figure.titlesize']=MAIN_TITLE_SIZE
from matplotlib.text import Text

def require(p):
    if not p.is_file(): raise FileNotFoundError(p)
    return p
def read(p): return json.loads(require(p).read_text())
def norm(x):return re.sub('[^a-z0-9]','',str(x).lower())
def col(df,*names):
    lookup={norm(c):c for c in df.columns}
    for name in names:
        if norm(name) in lookup:return lookup[norm(name)]
    raise ValueError(f'Missing {names}; available {list(df.columns)}')
def num(df,*names):return pd.to_numeric(df[col(df,*names)],errors='coerce')
def pct(x):
    x=np.asarray(x,dtype=float)
    return 100*x if np.nanmax(x)<=1.000001 else x
def panel(ax,letter,title):
    ax.set_title(title,fontsize=FONT_SIZE);ax.text(-.10,1.05,letter,transform=ax.transAxes,fontsize=FONT_SIZE,fontweight='bold')
def save(fig,stem):
    # Materialize ticks; main figure title 15 pt, all other text 12 pt.
    fig.canvas.draw()
    for txt in fig.findobj(match=Text):
        txt.set_fontfamily(RENDER_FONT)
        txt.set_fontsize(MAIN_TITLE_SIZE if txt is getattr(fig, '_suptitle', None) else FONT_SIZE)
    fig.canvas.draw()
    for ext in ['svg','png']:
        path=OUT/f'{stem}.{ext}'
        fig.savefig(path,bbox_inches='tight',facecolor='white',dpi=320)
        if ext=='svg': normalize_svg_font(path)
    plt.show();plt.close(fig)
    print('Saved:',OUT/f'{stem}.svg')
for p in [RUN/'evaluation_summary.json',TRAIN/'evaluation_summary.json',RUN/'atom_token_class_accuracy.csv',RUN/'molecule_diagnostics.csv',RUN/'objective_batches.csv',RUN/'cls_view0.npy',RUN/'projected_view0.npy',BENCH,UGI/'03_full_molecular_audit.json',UGI/'02_3d_index_census.json',LNP/'lnpdb_unique_with_ugi_overlap.csv',LNP/'summary.json']:
    print('FOUND' if p.is_file() else 'MISSING',p)
print('Output:',OUT)

## Figure 1 — General pretraining
Six main panels, nine plots. PCA alignment must be checked against the original evaluation manifest. No arbitrary manifest fallback.

In [ ]:
meta_candidates=[GENERAL/'validation_10k_seed17/sample_metadata.csv',GENERAL/'chemistry_slide_evidence/sample_metadata.csv',RUN/'sample_metadata.csv']
META=next((p for p in meta_candidates if p.is_file()),None) # change explicitly if necessary
print('PCA manifest:',META)
if META is None:raise FileNotFoundError(meta_candidates)
meta=pd.read_csv(META,keep_default_na=False)
if 'sample_order' not in meta:raise ValueError('Missing sample_order; verify alignment before plotting')
order=pd.to_numeric(meta.sample_order,errors='raise')
if order.duplicated().any():raise ValueError('Duplicate sample_order')
meta=meta.assign(sample_order=order).sort_values('sample_order',kind='stable').reset_index(drop=True)
arrs=[np.load(require(RUN/f),mmap_mode='r') for f in ['cls_view0.npy','projected_view0.npy']]
if not all(a.ndim==2 and len(a)==len(meta) for a in arrs):raise ValueError('Embedding/metadata row mismatch')
if not {'heavy_atoms','cLogP'}<=set(meta.columns):
    from rdkit import Chem
    from rdkit.Chem import Descriptors
    ms=[Chem.MolFromSmiles(s) for s in meta.smiles.astype(str)]
    meta['heavy_atoms']=[m.GetNumHeavyAtoms() if m is not None else np.nan for m in ms]
    meta['cLogP']=[Descriptors.MolLogP(m) if m is not None else np.nan for m in ms]
v=read(RUN/'evaluation_summary.json');t=read(TRAIN/'evaluation_summary.json')
atom=pd.read_csv(require(RUN/'atom_token_class_accuracy.csv'))
mol=pd.read_csv(require(RUN/'molecule_diagnostics.csv'));batch=pd.read_csv(require(RUN/'objective_batches.csv'))
fig=plt.figure(figsize=(27,22),layout='constrained');gs=fig.add_gridspec(3,2,height_ratios=[1,1.65,1])
a=fig.add_subplot(gs[0,0]);b=fig.add_subplot(gs[0,1]);x=np.arange(2)
for i,(key,label,color) in enumerate([('masked_atom_accuracy','Pretrained model',RED),('masked_atom_majority_class_baseline_accuracy','Majority-class baseline',GOLD)]):
    vals=pct([t[key],v[key]]);bars=a.bar(x+(i-.5)*.34,vals,.34,label=label,color=color);a.bar_label(bars,fmt='%.1f%%',padding=3)
a.set(xticks=x,xticklabels=['Training 10K','Validation 10K'],ylim=(0,125),ylabel='Accuracy (%)');a.legend(frameon=False,loc='upper center',bbox_to_anchor=(.5,1.0),ncol=2);panel(a,'a','Masked atom reconstruction')
vals=[float(v['coord_noisy_rmse']),float(v['coord_pred_rmse'])]
if vals[0]<=0:raise ValueError('Invalid noisy RMSE')
bars=b.bar(['Noisy input','Denoised output'],vals,color=[GOLD,RED],width=.55);b.bar_label(bars,fmt='%.4f',padding=3);b.set(ylabel='Aggregate coordinate RMSE (Å)',ylim=(0,max(vals)*1.25));b.text(.98,.90,f'{100*(1-vals[1]/vals[0]):.1f}% error reduction',transform=b.transAxes,ha='right');panel(b,'b','3D coordinate denoising — validation 10K')
sub=gs[1,0].subgridspec(2,2)
# Identical descriptor normalization across CLS and projection PCA; this is not potency.
PCA_NORMS={}
for prop in ['heavy_atoms','cLogP']:
    descriptor=pd.to_numeric(meta[prop],errors='coerce').to_numpy()
    finite=descriptor[np.isfinite(descriptor)]
    if finite.size==0: raise ValueError(f'No finite {prop}')
    PCA_NORMS[prop]=Normalize(vmin=float(finite.min()),vmax=float(finite.max()))
for i,(label,arr) in enumerate(zip(['CLS','Contrastive projection'],arrs)):
    pca=PCA(n_components=2,svd_solver='randomized',random_state=17);xy=pca.fit_transform(np.asarray(arr,dtype=np.float32))
    for j,prop in enumerate(['heavy_atoms','cLogP']):
        ax=fig.add_subplot(sub[i,j]);z=pd.to_numeric(meta[prop],errors='coerce').to_numpy();ok=np.isfinite(z)
        if not ok.any():raise ValueError(f'No valid {prop}')
        sc=ax.scatter(xy[ok,0],xy[ok,1],c=z[ok],s=2,alpha=.8,cmap=WARM,norm=PCA_NORMS[prop],linewidths=0,rasterized=False)
        ax.set(title=f'{label} — {prop}',xlabel=f'PC1 ({pca.explained_variance_ratio_[0]:.1%})',ylabel=f'PC2 ({pca.explained_variance_ratio_[1]:.1%})');ax.title.set_fontsize(FONT_SIZE);fig.colorbar(sc,ax=ax,shrink=.75)
        if i==j==0:ax.text(-.18,1.16,'c',transform=ax.transAxes,fontsize=FONT_SIZE,fontweight='bold')
d=fig.add_subplot(gs[1,1]);sym=col(atom,'token_symbol','element');n=num(atom,'masked_targets','n','count');acc=num(atom,'accuracy','masked_atom_accuracy')
# Keep elements through iodine as requested; omit Si/B/Na and any other rare categories.
keep=['C','N','O','S','F','Cl','Br','I']; symbols=atom[sym].astype(str).str.strip()
ok=symbols.isin(keep)&(n>0)&acc.notna()
print('Omitted atom classes:',atom.loc[~ok,[sym]].to_dict('records'))
if not ok.any(): raise ValueError('No requested elements found')
bars=d.bar(symbols[ok],pct(acc[ok]),color=[PALETTE[min(i,len(PALETTE)-1)] for i in range(int(ok.sum()))]);d.bar_label(bars,labels=[f'{int(k):,}' for k in n[ok]],padding=3,fontsize=FONT_SIZE)
d.set(ylabel='Recovery accuracy (%)',ylim=(0,115),xlabel='Element (labels above bars = masked targets)');panel(d,'d','Per-element masked atom recovery')
e=fig.add_subplot(gs[2,0]);x=num(mol,'coord_noisy_rmse');y=num(mol,'coord_pred_rmse');ok=x.notna()&y.notna()&(x>=0)&(y>=0)
if not ok.any():raise ValueError('No valid coordinate pairs')
e.scatter(x[ok],y[ok],s=5,alpha=.25,color=RED);mx=max(x[ok].max(),y[ok].max());e.plot([0,mx],[0,mx],'--',color='gray',lw=1);e.set(xlabel='Noisy input RMSE (Å)',ylabel='Denoised RMSE (Å)');panel(e,'e','Coordinate recovery per view')
f=fig.add_subplot(gs[2,1]);z=num(batch,'contrastive_loss','nt_xent_loss').dropna();f.hist(z,bins=30,color=PLUM);f.set(xlabel='Recorded NT-Xent loss',ylabel='Number of batches');panel(f,'f','Contrastive loss across batches')
fig.suptitle('CMRI_Model general pretraining validation',fontsize=MAIN_TITLE_SIZE,fontweight='bold');save(fig,'CMRI_Model_Figure1_general_pretraining')
print('PCA row count/order checked; confirm identity alignment against original evaluation.')

## Figure 2 — independent downstream and library panels
Use the completed Priority 1 and Priority 2 evidence, not the old composite PNG. The two Ugi summary panels are re-created from the original audit fields; the molecular structure is read from the representative record in the completed probe report. No pilot or full scan is launched.

In [ ]:
# Read cached library evidence only. Exact schemas from original notebooks.
full=read(UGI/'03_full_molecular_audit.json')
side=read(UGI/'02_3d_index_census.json')
if full.get('scope')!='FULL' or side.get('scope')!='FULL_VIEW_SCAN':
    raise ValueError('Completed full audit and sidecar census required')
total=int(full['audited_records']); unique=int(full['unique_canonical_molecules_hash_based']); dup=int(full['duplicate_excess_records']); cross=int(full['canonical_train_valid_intersection'])
if total<=0 or total!=int(full['expected_records']) or int(full['audited_shards'])!=int(full['expected_shards']) or total!=int(side['counted_records']):
    raise ValueError('Incomplete or inconsistent full audit')
n3={int(k):int(v) for k,v in full['n3d_histogram'].items()}
if sum(n3.values())!=total or any(k not in range(1,11) or v<0 for k,v in n3.items()):
    raise ValueError('Unexpected conformer histogram; inspect audit')
lnp=pd.read_csv(require(LNP/'lnpdb_unique_with_ugi_overlap.csv'))
def bool_column(s):
    z=s.astype(str).str.strip().str.lower().map({'true':True,'false':False,'1':True,'0':False})
    if z.isna().any():raise ValueError(f'Invalid boolean values in {s.name}')
    return z.astype(bool)
if lnp['canonical_smiles'].isna().any() or lnp['canonical_smiles'].duplicated().any():raise ValueError('Expected unique canonical LNPDB rows')
like=bool_column(lnp['ugi_like_core']); matched=bool_column(lnp['matched_ugi'])
counts=[int((like&matched).sum()),int((like&~matched).sum()),int((~like&matched).sum()),int((~like&~matched).sum())]
print('Ugi records:',total,'10-slot percentage:',100*n3.get(10,0)/total,'unique:',unique,'duplicates:',dup,'cross-split:',cross)
print('LNPDB overlap groups:',counts)
# Read precisely the actual representative from completed audit, not a newly generated structure.
import os, pickle
import lmdb
from rdkit import Chem
probe=read(UGI/'05_real_lmdb_probe_summary.json'); rep=probe['representative_real_structure']
LMDB_ROOT=Path(os.getenv('HEART_UGI_ROOT','/project/yli4/ypan/lipid_library/lmdb'))
record_path=require(LMDB_ROOT/rep['shard'])
key=str(rep['key']).encode('ascii')
env=lmdb.open(str(record_path),subdir=False,readonly=True,lock=False,readahead=False,max_readers=4)
try:
    with env.begin(write=False) as txn:
        payload=txn.get(key)
        if payload is None:raise KeyError(key)
        item=pickle.loads(payload) # trusted local laboratory LMDB only
finally:env.close()
atoms=list(item['atoms']); coords=np.asarray(item['coordinates'][0],dtype=float)
mol3=Chem.MolFromSmiles(item['smi'])
if mol3 is None:raise ValueError('Invalid representative SMILES')
mol3=Chem.AddHs(mol3)
if [a.GetSymbol() for a in mol3.GetAtoms()]!=atoms or coords.shape!=(len(atoms),3) or not np.isfinite(coords).all() or not 1<=int(rep['n3d'])<=10:
    raise ValueError('Representative atom/coordinate mismatch')
print('Actual conformer:',record_path,'key',rep['key'],'slot 0')


In [ ]:
def canonical_benchmark(value):
    return {'esol':'ESOL','delaney':'ESOL','freesolv':'FreeSolv','sampl':'FreeSolv','lipophilicity':'Lipophilicity','lipo':'Lipophilicity'}.get(norm(value))
def canonical_method(value):
    s=norm(value)
    if 'morgan' in s or 'ecfp' in s:return 'Morgan ECFP4'
    if 'rdkit' in s:return 'RDKit descriptors'
    if 'heavy' in s:return 'Heavy atoms only'
    if 'mean' in s or 'dummy' in s:return 'Mean label'
    if any(t in s for t in ['frozen','heart','pretrain','cls']):return 'Frozen pretrained CLS'
    return str(value).strip()
raw=pd.read_csv(require(BENCH));bc=col(raw,'benchmark','dataset','task');mc=col(raw,'model','method');fc=col(raw,'train_fraction','fraction');rc=col(raw,'rmse','test_rmse');sc=col(raw,'seed')
q=raw.assign(bench=raw[bc].map(canonical_benchmark),method=raw[mc].map(canonical_method),rmse=pd.to_numeric(raw[rc],errors='coerce'))
q=q[np.isclose(pd.to_numeric(q[fc],errors='coerce'),1.0)&q.bench.notna()&q.rmse.notna()].copy()
if q.duplicated(['bench','method',sc]).any():raise ValueError('Repeated seed rows; review before plotting')
agg=q.groupby(['bench','method']).rmse.agg(['mean','std','count']);display(agg)
methods=['Frozen pretrained CLS','RDKit descriptors','Morgan ECFP4','Heavy atoms only','Mean label']
# Functions shared by the composite and independent exports; no nested panel labels.
def benchmark_panel(ax,name):
    s=agg.loc[name].reindex(methods)
    if s['mean'].isna().any():raise ValueError(f'Missing benchmark methods: {name}')
    y=np.arange(len(s));ax.barh(y,s['mean'],xerr=s['std'].fillna(0),capsize=3,color=[RED,'#EACB78','#E6A34C','#DC8051','#A86661'],error_kw={'ecolor':'#493C36','lw':1})
    ax.set_yticks(y,['CMRI_Model (frozen CLS)','RDKit descriptors','Morgan ECFP4','Heavy atoms','Mean label'],fontsize=FONT_SIZE);ax.invert_yaxis();ax.set_xlabel('Scaffold-held-out RMSE',fontsize=FONT_SIZE);ax.set_xlim(0,max(s['mean']+s['std'].fillna(0))*1.60)
    for yy,(_,r) in enumerate(s.iterrows()):
        sd=r['std'];err=sd if np.isfinite(sd) else 0
        ax.text(r['mean']+err+ax.get_xlim()[1]*.015,yy,f"{r['mean']:.3f} ± {sd:.3f}" if np.isfinite(sd) else f"{r['mean']:.3f} (n=1)",va='center',fontsize=FONT_SIZE)
    ax.set_title(name,fontsize=FONT_SIZE,pad=8)
def conformer_panel(ax):
    xs=np.arange(1,11);ys=np.array([n3.get(int(k),0) for k in xs])/total*100
    ax.bar(xs,ys,color=GOLD,width=.75);ax.set(xticks=xs,ylim=(0,110),xlabel='Valid 3D conformers',ylabel='Records (%)',title='Ugi-4CR conformer count')
    ax.text(10,ys[-1]+2,f'{ys[-1]:.1f}%',ha='center',fontsize=FONT_SIZE)
def identity_panel(ax):
    values=[unique,dup,cross];ax.barh(['Unique identities','Duplicate excess','Train–val overlap'],values,color=[GOLD,RED,PLUM],height=.55);ax.invert_yaxis();ax.set(title='Canonical identity',xlabel='Molecular identities / records',xlim=(0,max(values)*1.42))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x,pos:f'{x/1e6:.0f}M' if x>=1e6 else f'{x:,.0f}'))
    for i,v in enumerate(values):ax.text(v+max(values)*.018,i,f'{v:,}',va='center',fontsize=FONT_SIZE)
def structure_panel(ax):
    # Element colors are chemical identifiers, not a quantitative palette.
    atom_colors={'C':'#74665F','H':'#D9D3CC','N':'#7C546D','O':'#C95752','S':'#E6A34C','P':'#A66045'}
    for bond in mol3.GetBonds():
        i,j=bond.GetBeginAtomIdx(),bond.GetEndAtomIdx();ax.plot(*coords[[i,j]].T,color='#A99B8C',lw=.8,alpha=.8)
    ax.scatter(*coords.T,c=[atom_colors.get(a,'#7C546D') for a in atoms],s=[30 if a!='H' else 7 for a in atoms],depthshade=False,edgecolors='none')
    # Restore the original physical coordinate frame, with a small margin in Å.
    ax.set_axis_on()
    lo=coords.min(axis=0);hi=coords.max(axis=0)
    span=np.maximum(hi-lo,.01);pad=np.maximum(.07*span,.35)
    bounds=list(zip(lo-pad,hi+pad))
    ax.set_xlim(*bounds[0]);ax.set_ylim(*bounds[1]);ax.set_zlim(*bounds[2])
    ax.set_box_aspect([b-a for a,b in bounds]);ax.view_init(elev=24,azim=-60)
    ax.set_xlabel('X (Å)',labelpad=6);ax.set_ylabel('Y (Å)',labelpad=6);ax.set_zlabel('Z (Å)',labelpad=6)
    from matplotlib.ticker import MaxNLocator
    for axis in (ax.xaxis,ax.yaxis,ax.zaxis):
        axis.set_major_locator(MaxNLocator(nbins=4))
        axis.pane.set_facecolor((.96,.95,.94,.35))
        axis.pane.set_edgecolor('#A99B8C')
        axis.line.set_color('#665B54')
    ax.tick_params(axis='both',labelsize=FONT_SIZE,pad=1)
    ax.grid(True)
    ax.set_title('Stored 3D conformer',fontsize=FONT_SIZE,pad=8)
def overlap_panel(ax):
    labels=['Ugi-like\nmatched','Ugi-like\nnovel','Other\nmatched','Other\nnovel'];bars=ax.bar(labels,counts,color=[GOLD,'#DC8051',GOLD,RED]);ax.bar_label(bars,labels=[f'{v:,}' for v in counts],padding=3,fontsize=FONT_SIZE);ax.set(title='LNPDB–Ugi-4CR overlap',ylabel='Unique canonical IL molecules',ylim=(0,max(counts)*1.18))
# Fixed axes positions avoid constrained-layout conflicts between the 3- and 4-column rows.
# The wider horizontal gaps leave room for each panel's tick labels and annotations.
def tag(ax,letter):
    p=ax.get_position(original=True)
    ax.figure.text(p.x0-.018,p.y1+.025,letter,fontsize=FONT_SIZE,fontweight='bold',ha='left',va='center')
def compact(ax):
    ax.title.set_fontsize(FONT_SIZE)
    ax.title.set_fontweight('bold')
    ax.title.set_position((.5,1.0))
    ax.tick_params(axis='both',labelsize=FONT_SIZE,pad=3)
    ax.xaxis.label.set_size(FONT_SIZE);ax.yaxis.label.set_size(FONT_SIZE)
    return ax
fig=plt.figure(figsize=(34,19),layout=None)
# [left, bottom, width, height] as fractions of the whole figure.
TOP=[(.115,.575,.215,.31),(.425,.575,.215,.31),(.735,.575,.215,.31)]
BOTTOM=[(.055,.115,.19,.30),(.335,.115,.18,.30),(.555,.115,.19,.30),(.805,.115,.165,.30)]
for i,(name,pos) in enumerate(zip(['FreeSolv','ESOL','Lipophilicity'],TOP)):
    ax=fig.add_axes(pos);benchmark_panel(ax,name);compact(ax);tag(ax,'abc'[i])
for j,(fn,pos) in enumerate(zip([conformer_panel,identity_panel,structure_panel,overlap_panel],BOTTOM)):
    ax=fig.add_axes(pos,**({'projection':'3d'} if j==2 else {}));fn(ax);compact(ax)
    if j==2:
        # Shrink only the visual projection, preserving the stored coordinates and aspect ratio.
        try:ax.set_box_aspect([np.diff(ax.get_xlim())[0],np.diff(ax.get_ylim())[0],np.diff(ax.get_zlim())[0]],zoom=.86)
        except TypeError:pass # older Matplotlib lacks zoom
    tag(ax,'defg'[j])
fig.suptitle('CMRI_Model downstream transfer and lipid-library characterization',fontsize=MAIN_TITLE_SIZE,fontweight='bold',y=.975)
save(fig,'CMRI_Model_Figure2_downstream_and_library')
# Independent, editable SVG panels for Illustrator; titles but no internal panel letters.
for stem,fn,is3d,size in [('d_ugi_conformers',conformer_panel,False,(9,7.5)),('e_ugi_identity',identity_panel,False,(11,7.5)),('f_real_conformer',structure_panel,True,(9,7.5)),('g_lnpdb_overlap',overlap_panel,False,(9,7.5))]:
    ff=plt.figure(figsize=size,layout='constrained');ax=ff.add_subplot(111,**({'projection':'3d'} if is3d else {}));fn(ax);save(ff,'CMRI_Model_'+stem)
(OUT/'figure2_sources.json').write_text(json.dumps({'full_audit':str(UGI/'03_full_molecular_audit.json'),'sidecar':str(UGI/'02_3d_index_census.json'),'overlap':str(LNP/'lnpdb_unique_with_ugi_overlap.csv'),'benchmark':str(BENCH),'lmdb':str(record_path),'key':rep['key'],'slot':0,'conformer_count_source':'full audit metadata; not all payloads rechecked'},indent=2))
